# trainsafe — CUDA verification

**Runtime → Change runtime type → T4 GPU** before running.

Run all cells top to bottom. Each test prints PASSED or FAILED at the end.

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU — go to Runtime > Change runtime type > T4 GPU"
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"CUDA: {torch.version.cuda}")

In [ ]:
!pip install "trainsafe[trl] @ git+https://github.com/ammarhassona/trainsafe.git" datasets langdetect -q

In [ ]:
# Test 1: generate_output runs on GPU and returns a string
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from trainsafe.utils import generate_output

MODEL_ID = "trl-internal-testing/tiny-Qwen2ForCausalLM-2.5"
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16).cuda()
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

prompt_ids = tokenizer("Hello, how are you?", return_tensors="pt")["input_ids"][0]
output = generate_output(model, tokenizer, prompt_ids, max_new_tokens=32)

assert isinstance(output, str) and len(output) > 0
print(f"Test 1 PASSED — output: {output!r}")

In [ ]:
# Test 2: CUDA cache is cleared after inference (no persistent memory leak)
torch.cuda.empty_cache()
before = torch.cuda.memory_allocated()

prompt_ids = tokenizer("Test prompt", return_tensors="pt")["input_ids"][0]
generate_output(model, tokenizer, prompt_ids, max_new_tokens=64)

torch.cuda.empty_cache()
after = torch.cuda.memory_allocated()
diff_mb = (after - before) / 1e6

assert diff_mb < 10, f"Memory grew by {diff_mb:.1f}MB after cache clear"
print(f"Test 2 PASSED — memory delta after cache clear: {diff_mb:.2f}MB")

In [ ]:
# Test 3: long prompts are truncated instead of erroring
original_max = tokenizer.model_max_length
tokenizer.model_max_length = 64

long_prompt = "word " * 200
prompt_ids = tokenizer(long_prompt, return_tensors="pt")["input_ids"][0]
output = generate_output(model, tokenizer, prompt_ids, max_new_tokens=32)

tokenizer.model_max_length = original_max
assert isinstance(output, str)
print(f"Test 3 PASSED — long prompt truncated cleanly, output: {output!r}")

In [ ]:
# Test 4: full SFTTrainer run on GPU — callback fires, checks run, no crash
from datasets import load_dataset
from trl import SFTConfig, SFTTrainer
from trainsafe import TrainSafeCallback

train_model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float32).cuda()
ds = load_dataset("trl-internal-testing/zen", "standard_prompt_completion")

trainer = SFTTrainer(
    model=train_model,
    args=SFTConfig(
        output_dir="/tmp/trainsafe_cuda_sft",
        eval_strategy="steps",
        eval_steps=5,
        max_steps=5,
        per_device_train_batch_size=2,
        per_device_eval_batch_size=2,
        report_to="none",
    ),
    train_dataset=ds["train"],
    eval_dataset=ds["test"],
    callbacks=[TrainSafeCallback(num_inference_samples=2, max_new_tokens=32, log_to_wandb=False)],
)
trainer.train()
print("Test 4 PASSED — SFTTrainer with trainsafe ran on GPU without errors")

In [ ]:
print("All 4 CUDA tests passed.")